In [4]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from google.colab import drive
drive.mount('/content/drive')

marketing_data = pd.read_csv('/content/drive/MyDrive/Marketing.csv')


# ==========================================
# 1. Custom Transformer para Datas
# ==========================================
class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        # Converte para datetime se não for
        if not np.issubdtype(X_copy['c_date'].dtype, np.datetime64):
            X_copy['c_date'] = pd.to_datetime(X_copy['c_date'])

        X_copy['year'] = X_copy['c_date'].dt.year
        X_copy['month'] = X_copy['c_date'].dt.month
        X_copy['day_of_week'] = X_copy['c_date'].dt.dayofweek
        X_copy['is_weekend'] = X_copy['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

        return X_copy.drop(columns=['c_date'])

# ==========================================
# 2. Carga e Preparação
# ==========================================
# Supondo que o arquivo já esteja no ambiente
df = pd.read_csv('/content/drive/MyDrive/Marketing.csv')

# Features e Target
X = df.drop(columns=['revenue', 'id', 'campaign_id']) # Removemos IDs que não ajudam na predição
y = df['revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 3. Definição do Pipeline Profissional
# ==========================================

# Colunas numéricas e categóricas
numeric_features = ['mark_spent', 'impressions', 'clicks', 'leads', 'orders']
# As colunas de data serão geradas, então não listamos aqui, o DateExtractor cuida delas
categorical_features = ['campaign_name', 'category']

# Preprocessamento
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        # As datas extraídas (year, month, etc) passarão "passthrough" ou precisarão de scaler
        # Para simplificar neste exemplo, focamos nas principais.
    ],
    remainder='passthrough' # Deixa passar as colunas de data criadas
)

# Pipeline Completo
pipeline = Pipeline([
    ('date_extractor', DateFeatureExtractor()), # 1. Extrai datas
    ('preprocessor', preprocessor),             # 2. Trata números e categorias
    ('model', RandomForestRegressor(random_state=42)) # 3. Modelo
])

# ==========================================
# 4. Treinamento e Otimização (GridSearch)
# ==========================================
# Otimização leve para portfólio
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5]
}

print("Iniciando treinamento e otimização...")
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

# ==========================================
# 5. Avaliação Final
# ==========================================
y_pred = best_model.predict(X_test)

print("\n--- Métricas Finais ---")
print(f"Melhores Parâmetros: {grid_search.best_params_}")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

Mounted at /content/drive
Iniciando treinamento e otimização...

--- Métricas Finais ---
Melhores Parâmetros: {'model__max_depth': 10, 'model__min_samples_split': 2, 'model__n_estimators': 100}
R² Score: 0.9145
MAE: 29964.50
RMSE: 61795.83


In [5]:
# ==========================================
# 6. Exportação do Modelo (Pipeline Completo)
# ==========================================
import os

# Salva o pipeline inteiro (inclui a lógica de data, encoders e o modelo)
model_filename = 'revenue_prediction_pipeline.pkl'
joblib.dump(best_model, model_filename)

print(f"\n✅ Modelo exportado com sucesso como: {model_filename}")
print("Este arquivo contém toda a lógica de pré-processamento e predição.")


✅ Modelo exportado com sucesso como: revenue_prediction_pipeline.pkl
Este arquivo contém toda a lógica de pré-processamento e predição.


In [7]:
import joblib

# Carregar o pipeline completo
loaded_pipeline = joblib.load('revenue_prediction_pipeline.pkl')

print("✅ Pipeline de predição carregado com sucesso!")

✅ Pipeline de predição carregado com sucesso!


### 2. Fazer Previsões com Novos Dados

Para fazer previsões, você precisará de novos dados no mesmo formato dos seus dados originais de treinamento (excluindo as colunas `revenue`, `id` e `campaign_id`).

Vamos criar um exemplo de novos dados para demonstração:

### 3. Baixar o Modelo Treinado

Você pode fazer o download do arquivo `.pkl` diretamente do Colab para o seu computador usando a função `google.colab.files.download`:

In [9]:
from google.colab import files

# Nome do arquivo do modelo
model_filename = 'revenue_prediction_pipeline.pkl'

# Tentar fazer o download do arquivo
try:
    files.download(model_filename)
    print(f"✅ Download de '{model_filename}' iniciado.")
except Exception as e:
    print(f"❌ Erro ao tentar baixar o arquivo '{model_filename}': {e}")
    print("Certifique-se de que o arquivo existe e que a célula foi executada após a exportação do modelo.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download de 'revenue_prediction_pipeline.pkl' iniciado.


In [8]:
import pandas as pd

# Exemplo de novos dados (sem a coluna 'revenue' e IDs)
new_data = pd.DataFrame({
    'c_date': ['2023-01-15'],
    'campaign_name': ['google_hot'],
    'category': ['search'],
    'impressions': [150000],
    'mark_spent': [5000.00],
    'clicks': [1500],
    'leads': [100],
    'orders': [50]
})

print("Novos dados para predição:")
display(new_data)

# Fazer a predição usando o pipeline carregado
new_predictions = loaded_pipeline.predict(new_data)

print(f"\nPrevisão de Receita para os novos dados: ${new_predictions[0]:.2f}")

Novos dados para predição:


,c_date,campaign_name,category,impressions,mark_spent,clicks,leads,orders
0,2023-01-15,google_hot,search,150000,5000.0,1500,100,50



Previsão de Receita para os novos dados: $290838.13


### 1. Preparar o Ambiente para Flask

Primeiro, precisamos instalar o Flask, caso ainda não esteja instalado.

In [10]:
!pip install Flask pandas joblib

### 2. Criar a API Flask

Este código define uma aplicação Flask que:
*   Carrega o modelo `revenue_prediction_pipeline.pkl` quando a aplicação é iniciada.
*   Cria um endpoint `/predict` que aceita requisições POST com dados JSON.
*   Converte o JSON em um DataFrame do Pandas e usa o modelo para fazer uma previsão.
*   Retorna a previsão como JSON.

In [11]:
import joblib
import pandas as pd
from flask import Flask, request, jsonify
import threading
import time

app = Flask(__name__)

# Carregar o modelo uma vez quando o aplicativo Flask é iniciado
try:
    model_path = 'revenue_prediction_pipeline.pkl'
    loaded_pipeline = joblib.load(model_path)
    print(f"✅ Modelo '{model_path}' carregado com sucesso para a API Flask.")
except Exception as e:
    loaded_pipeline = None
    print(f"❌ Erro ao carregar o modelo: {e}")

@app.route('/predict', methods=['POST'])
def predict():
    if loaded_pipeline is None:
        return jsonify({'error': 'Modelo não carregado. Verifique os logs do servidor.'}), 500

    try:
        data = request.get_json(force=True)

        # Converter o JSON recebido em um DataFrame
        # Assumimos que o JSON é uma lista de dicionários, ou um único dicionário
        if isinstance(data, dict):
            new_data_df = pd.DataFrame([data])
        elif isinstance(data, list):
            new_data_df = pd.DataFrame(data)
        else:
            return jsonify({'error': 'Formato de entrada JSON inválido. Esperado objeto ou array de objetos.'}), 400

        # Certifique-se de que as colunas estão na ordem esperada, se necessário
        # Para este modelo, a ordem é importante devido ao ColumnTransformer e DateFeatureExtractor
        # As colunas devem ser as mesmas que X_train (exceto 'id', 'campaign_id', 'revenue')
        # 'c_date', 'campaign_name', 'category', 'impressions', 'mark_spent', 'clicks', 'leads', 'orders'
        expected_columns = ['c_date', 'campaign_name', 'category', 'impressions', 'mark_spent', 'clicks', 'leads', 'orders']
        missing_cols = [col for col in expected_columns if col not in new_data_df.columns]
        if missing_cols:
            return jsonify({'error': f'Colunas ausentes na entrada: {missing_cols}'}), 400

        # Reordenar as colunas para garantir consistência
        new_data_df = new_data_df[expected_columns]

        predictions = loaded_pipeline.predict(new_data_df)
        return jsonify({'predictions': predictions.tolist()})

    except Exception as e:
        return jsonify({'error': str(e)}), 500

# Função para executar o Flask em um thread separado
def run_flask_app():
    # Usando '0.0.0.0' para ser acessível de fora do localhost no ambiente Colab
    app.run(host='0.0.0.0', port=5000, debug=False)

# Iniciar o servidor Flask em um thread separado
flask_thread = threading.Thread(target=run_flask_app)
flask_thread.daemon = True # Permite que o programa principal saia mesmo que o thread esteja rodando
flask_thread.start()

print("\n✅ Servidor Flask iniciado em segundo plano na porta 5000.")
print("Para testar, você precisará de uma nova célula ou de uma ferramenta como 'ngrok' para expor a porta 5000.")
print("Aguarde alguns segundos para que o servidor inicialize completamente...")


✅ Modelo 'revenue_prediction_pipeline.pkl' carregado com sucesso para a API Flask.

✅ Servidor Flask iniciado em segundo plano na porta 5000.
Para testar, você precisará de uma nova célula ou de uma ferramenta como 'ngrok' para expor a porta 5000.
Aguarde alguns segundos para que o servidor inicialize completamente...


### 3. Testar a API (no Colab)

Para testar esta API aqui no Colab, você precisaria de um cliente HTTP. Usaremos a biblioteca `requests` do Python. Como o servidor Flask está rodando em um thread separado, podemos enviar requisições para ele.

**Nota**: Se você quisesse acessar esta API de fora do Colab, precisaria de uma ferramenta como `ngrok` para criar um túnel para a porta 5000.

In [23]:
import requests
import json
import time

# Aguardar um pouco para garantir que o servidor Flask esteja pronto
time.sleep(5)

# Dados de exemplo para enviar à API
example_data = {
    'c_date': '2023-02-20',
    'campaign_name': 'google_wide',
    'category': 'search',
    'impressions': 200000,
    'mark_spent': 7500.50,
    'clicks': 2000,
    'leads': 120,
    'orders': 60
}

# URL do endpoint de predição
# No Colab, 'localhost' ou '127.0.0.1' funciona para comunicação interna
# Agora usando a URL do ngrok para acesso externo
predict_url = 'https://scintillant-samuel-dowily.ngrok-free.dev/predict'

print(f"Enviando dados para a API: {example_data}")

try:
    # Enviar a requisição POST
    response = requests.post(predict_url, json=example_data)

    # Verificar a resposta
    if response.status_code == 200:
        result = response.json()
        print(f"\n✅ Previsão recebida: {result['predictions'][0]:.2f}")
    else:
        print(f"\n❌ Erro na requisição. Código de status: {response.status_code}")
        print(f"Mensagem de erro: {response.json().get('error', 'Nenhuma mensagem de erro.')}")
except requests.exceptions.ConnectionError as e:
    print(f"\n❌ Erro de conexão: {e}")
    print("Certifique-se de que o servidor Flask está rodando na porta 5000 e o túnel ngrok está ativo.")
except Exception as e:
    print(f"\n❌ Um erro inesperado ocorreu: {e}")

INFO:werkzeug:127.0.0.1 - - [05/Jan/2026 19:02:22] "POST /predict HTTP/1.1" 200 -


Enviando dados para a API: {'c_date': '2023-02-20', 'campaign_name': 'google_wide', 'category': 'search', 'impressions': 200000, 'mark_spent': 7500.5, 'clicks': 2000, 'leads': 120, 'orders': 60}

✅ Previsão recebida: 264749.94


# Task
Instalar `pyngrok`, autenticar o ngrok com o authtoken fornecido, criar um túnel ngrok para a porta 5000 onde a aplicação Flask está rodando, e obter a URL pública para acesso externo à API Flask.

## Instalar pyngrok

### Subtask:
Instalar a biblioteca `pyngrok` que permite interagir com o ngrok em Python.


**Reasoning**:
Install the `pyngrok` library using pip.



In [13]:
print("Instalando pyngrok...")
!pip install pyngrok
print("pyngrok instalado com sucesso.")

Instalando pyngrok...
pyngrok instalado com sucesso.


## Autenticar ngrok

### Subtask:
Configurar o ngrok com um authtoken.


**Reasoning**:
The next step is to configure ngrok with an authtoken, which involves importing the ngrok function and using `set_auth_token` with a placeholder for the user's actual token.



In [21]:
from pyngrok import ngrok

# Configure o ngrok com seu authtoken
# Substitua 'YOUR_NGROK_AUTHTOKEN' pelo seu token real
# Você pode encontrar seu authtoken em ngrok.com/dashboard/your-authtoken
ngrok.set_auth_token("37qmyQ1o1UwVdc3wR7WiD5eQInD_7Pg2EQkJzkK2Zwb18pGXR")

print("✅ ngrok authtoken configurado (placeholder). Por favor, substitua pelo seu token real.")

✅ ngrok authtoken configurado (placeholder). Por favor, substitua pelo seu token real.


A célula anterior configurou um placeholder para o `authtoken` do ngrok. Para que o ngrok funcione corretamente, **você deve substituir `"YOUR_NGROK_AUTHTOKEN"` no código da célula `4ad4bf06` pelo seu authtoken real**. Você pode encontrar seu authtoken em [ngrok.com/dashboard/your-authtoken](https://ngrok.com/dashboard/your-authtoken).

Após a substituição, execute novamente a célula `4ad4bf06` para aplicar a configuração.

## Criar o Túnel ngrok

### Subtask:
Criar um túnel ngrok para a porta 5000 da aplicação Flask e obter a URL pública.

**Reasoning**:
Now that the subtask for creating the ngrok tunnel has been introduced, the next step is to provide the Python code to establish the tunnel to port 5000 and display the public URL.



In [22]:
from pyngrok import ngrok

# Abrir um t\u00fanel ngrok para a porta 5000
# ngrok.connect() ir\u00e1 baixar o bin\u00e1rio do ngrok se ainda n\u00e3o estiver presente
public_url = ngrok.connect(5000)

print(f"\u2705 ngrok Tunnel URL: {public_url}")
print("Use esta URL p\u00fablica para acessar sua API Flask de fora do Colab.")

✅ ngrok Tunnel URL: NgrokTunnel: "https://scintillant-samuel-dowily.ngrok-free.dev" -> "http://localhost:5000"
Use esta URL pública para acessar sua API Flask de fora do Colab.


# Task
Configurar um ambiente React para o projeto. Inclui os passos para inicializar um novo projeto React usando `create-react-app` ou Vite e instalar as dependências básicas.

## Configurar o Ambiente React

### Subtask:
Inicializar um novo projeto React usando Vite e instalar as dependências básicas.


## Configurar o Ambiente React

### Subtask:
Inicializar um novo projeto React usando Vite e instalar as dependências básicas.

#### Instructions
1.  Abra seu terminal ou prompt de comando local (fora do ambiente Colab).
2.  Crie um novo projeto React usando Vite executando o comando: `npm create vite@latest meu-app-react -- --template react`. Você pode substituir `meu-app-react` pelo nome que desejar para seu projeto.
3.  Navegue até o diretório do seu novo projeto: `cd meu-app-react`.
4.  Instale as dependências básicas do projeto executando: `npm install`.
5.  Inicie o servidor de desenvolvimento para verificar se tudo está funcionando: `npm run dev`. Isso deve abrir seu aplicativo React em seu navegador padrão.
6.  Pressione `Ctrl+C` no terminal para parar o servidor de desenvolvimento, pois continuaremos a configurar o aplicativo.

## Configurar o Ambiente React

### Subtask:
Inicializar um novo projeto React usando Vite e instalar as dependências básicas.

#### Instructions
1.  Abra seu terminal ou prompt de comando local (fora do ambiente Colab).
2.  Crie um novo projeto React usando Vite executando o comando: `npm create vite@latest meu-app-react -- --template react`. Você pode substituir `meu-app-react` pelo nome que desejar para seu projeto.
3.  Navegue até o diretório do seu novo projeto: `cd meu-app-react`.
4.  Instale as dependências básicas do projeto executando: `npm install`.
5.  Inicie o servidor de desenvolvimento para verificar se tudo está funcionando: `npm run dev`. Isso deve abrir seu aplicativo React em seu navegador padrão.
6.  Pressione `Ctrl+C` no terminal para parar o servidor de desenvolvimento, pois continuaremos a configurar o aplicativo.
7.  Abra o diretório `meu-app-react` em seu editor de código preferido (como VS Code) para começar a editar os arquivos do projeto.

## Configurar o Ambiente React

### Subtask:
Inicializar um novo projeto React usando Vite e instalar as dependências básicas.

#### Instructions
1.  Abra seu terminal ou prompt de comando local (fora do ambiente Colab).
2.  Crie um novo projeto React usando Vite executando o comando: `npm create vite@latest meu-app-react -- --template react`. Você pode substituir `meu-app-react` pelo nome que desejar para seu projeto.
3.  Navegue até o diretório do seu novo projeto: `cd meu-app-react`.
4.  Instale as dependências básicas do projeto executando: `npm install`.
5.  Inicie o servidor de desenvolvimento para verificar se tudo está funcionando: `npm run dev`. Isso deve abrir seu aplicativo React em seu navegador padrão.
6.  Pressione `Ctrl+C` no terminal para parar o servidor de desenvolvimento, pois continuaremos a configurar o aplicativo.
7.  Abra o diretório `meu-app-react` em seu editor de código preferido (como VS Code) para começar a editar os arquivos do projeto.

## Criar Componente de Formulário de Entrada

### Subtask:
Desenvolver um componente React que inclua campos de entrada para cada uma das variáveis necessárias para a predição (c_date, campaign_name, category, impressions, mark_spent, clicks, leads, orders), permitindo que o usuário insira novos dados.


```markdown
### **PredictionForm.jsx**

No diretório `src` do seu projeto React, crie um novo arquivo chamado `PredictionForm.jsx` e adicione o seguinte código:

```jsx
import React, { useState } from 'react';

function PredictionForm() {
  const [c_date, setC_date] = useState('');
  const [campaign_name, setCampaign_name] = useState('');
  const [category, setCategory] = useState('');
  const [impressions, setImpressions] = useState('');
  const [mark_spent, setMark_spent] = useState('');
  const [clicks, setClicks] = useState('');
  const [leads, setLeads] = useState('');
  const [orders, setOrders] = useState('');

  const handleSubmit = (e) => {
    e.preventDefault();
    // A lógica de submissão (enviar para a API) será implementada na próxima etapa.
    const formData = {
      c_date,
      campaign_name,
      category,
      impressions: parseFloat(impressions),
      mark_spent: parseFloat(mark_spent),
      clicks: parseInt(clicks, 10),
      leads: parseInt(leads, 10),
      orders: parseInt(orders, 10),
    };
    console.log('Dados do formulário:', formData);
    alert('Dados enviados para o console. A funcionalidade da API será adicionada em breve.');
  };

  return (
    <div style={{ maxWidth: '600px', margin: '50px auto', padding: '20px', border: '1px solid #ccc', borderRadius: '8px', boxShadow: '0 2px 4px rgba(0,0,0,0.1)' }}>
      <h2>Previsão de Receita de Marketing</h2>
      <form onSubmit={handleSubmit} style={{ display: 'grid', gridTemplateColumns: '1fr 1fr', gap: '15px' }}>
        <div>
          <label htmlFor="c_date" style={{ display: 'block', marginBottom: '5px' }}>Data da Campanha:</label>
          <input
            type="date"
            id="c_date"
            value={c_date}
            onChange={(e) => setC_date(e.target.value)}
            required
            style={{ width: '90%', padding: '8px', border: '1px solid #ddd', borderRadius: '4px' }}
          />
        </div>
        <div>
          <label htmlFor="campaign_name" style={{ display: 'block', marginBottom: '5px' }}>Nome da Campanha:</label>
          <input
            type="text"
            id="campaign_name"
            value={campaign_name}
            onChange={(e) => setCampaign_name(e.target.value)}
            required
            style={{ width: '90%', padding: '8px', border: '1px solid #ddd', borderRadius: '4px' }}
          />
        </div>
        <div>
          <label htmlFor="category" style={{ display: 'block', marginBottom: '5px' }}>Categoria:</label>
          <input
            type="text"
            id="category"
            value={category}
            onChange={(e) => setCategory(e.target.value)}
            required
            style={{ width: '90%', padding: '8px', border: '1px solid #ddd', borderRadius: '4px' }}
          />
        </div>
        <div>
          <label htmlFor="impressions" style={{ display: 'block', marginBottom: '5px' }}>Impressões:</label>
          <input
            type="number"
            id="impressions"
            value={impressions}
            onChange={(e) => setImpressions(e.target.value)}
            required
            style={{ width: '90%', padding: '8px', border: '1px solid #ddd', borderRadius: '4px' }}
          />
        </div>
        <div>
          <label htmlFor="mark_spent" style={{ display: 'block', marginBottom: '5px' }}>Gasto de Marketing:</label>
          <input
            type="number"
            id="mark_spent"
            step="0.01"
            value={mark_spent}
            onChange={(e) => setMark_spent(e.target.value)}
            required
            style={{ width: '90%', padding: '8px', border: '1px solid #ddd', borderRadius: '4px' }}
          />
        </div>
        <div>
          <label htmlFor="clicks" style={{ display: 'block', marginBottom: '5px' }}>Cliques:</label>
          <input
            type="number"
            id="clicks"
            value={clicks}
            onChange={(e) => setClicks(e.target.value)}
            required
            style={{ width: '90%', padding: '8px', border: '1px solid #ddd', borderRadius: '4px' }}
          />
        </div>
        <div>
          <label htmlFor="leads" style={{ display: 'block', marginBottom: '5px' }}>Leads:</label>
          <input
            type="number"
            id="leads"
            value={leads}
            onChange={(e) => setLeads(e.target.value)}
            required
            style={{ width: '90%', padding: '8px', border: '1px solid #ddd', borderRadius: '4px' }}
          />
        </div>
        <div>
          <label htmlFor="orders" style={{ display: 'block', marginBottom: '5px' }}>Pedidos:</label>
          <input
            type="number"
            id="orders"
            value={orders}
            onChange={(e) => setOrders(e.target.value)}
            required
            style={{ width: '90%', padding: '8px', border: '1px solid #ddd', borderRadius: '4px' }}
          />
        </div>
        <button
          type="submit"
          style={{
            gridColumn: '1 / -1',
            padding: '10px 15px',
            backgroundColor: '#007bff',
            color: 'white',
            border: 'none',
            borderRadius: '4px',
            cursor: 'pointer',
            fontSize: '16px',
            marginTop: '10px'
          }}
        >
          Obter Previsão
        </button>
      </form>
    </div>
  );
}

export default PredictionForm;
```

**Explicação:**

*   **`useState` Hooks:** Para cada campo de entrada, um estado individual é criado para gerenciar seu valor. Isso garante que o componente seja re-renderizado quando o valor de um campo muda, mantendo o formulário reativo.
*   **Elementos `<input>`:** São definidos com tipos apropriados (`date`, `text`, `number`) para otimizar a experiência do usuário e garantir a validação básica. O atributo `step="0.01"` é adicionado para `mark_spent` para permitir valores decimais.
*   **`value` e `onChange`:** Cada `input` está 'controlado' pelo React, o que significa que seu valor é obtido do estado (`value={c_date}`) e qualquer alteração no campo atualiza o estado correspondente (`onChange={(e) => setC_date(e.target.value)}`).
*   **`handleSubmit`:** Uma função de placeholder para o `onSubmit` do formulário. Por enquanto, ela apenas imprime os dados do formulário no console e exibe um alerta. A integração com a API Flask será feita na próxima etapa.
*   **Estilização:** Alguns estilos inline básicos foram adicionados para tornar o formulário mais apresentável.

### **Atualizar `App.jsx`**

Agora, abra o arquivo `App.jsx` (ou o arquivo principal do seu aplicativo React, que geralmente está no diretório `src`) e substitua seu conteúdo pelo seguinte código para importar e renderizar o `PredictionForm`:

```jsx
import PredictionForm from './PredictionForm'; // Importe o componente que você acabou de criar
import './App.css'; // Mantenha a importação de CSS se quiser usar o padrão ou seus próprios estilos

function App() {
  return (
    <div className="App">
      <PredictionForm /> {/* Renderize o componente do formulário aqui */}
    </div>
  );
}

export default App;
```

**Explicação:**

*   **`import PredictionForm from './PredictionForm';`**: Esta linha importa o componente `PredictionForm` do arquivo `PredictionForm.jsx` para o `App.jsx`.
*   **`<PredictionForm />`**: Dentro do `return` do componente `App`, estamos renderizando o `PredictionForm`. Isso fará com que o formulário apareça na sua aplicação web quando você a iniciar (`npm run dev` ou `yarn dev`).

Após essas alterações, salve ambos os arquivos (`PredictionForm.jsx` e `App.jsx`) e inicie seu servidor de desenvolvimento React (se ainda não estiver rodando). Você deverá ver o formulário de entrada no seu navegador.

## Integrar com a API Flask

### Subtask:
Implementar a lógica no componente React para fazer requisições POST para a URL pública do ngrok (ou a URL da sua API Flask implantada) com os dados inseridos pelo usuário. Isso envolverá o uso de `fetch` ou `axios` para enviar o JSON e receber a previsão.


```markdown
Para integrar com a API Flask a partir de um componente React, você precisará criar ou modificar um arquivo `.jsx` (por exemplo, `PredictionForm.jsx`) em seu projeto React. Este código não pode ser executado diretamente aqui no Colab, mas é o modelo que você deve usar em seu ambiente de desenvolvimento React.

---

## `src/components/PredictionForm.jsx` (Exemplo de Implementação)

Crie ou modifique o arquivo `src/components/PredictionForm.jsx` com o seguinte código. Certifique-se de substituir `YOUR_NGROK_PUBLIC_URL_HERE` pela URL pública do ngrok que foi gerada na célula anterior (por exemplo, `https://scintillant-samuel-dowily.ngrok-free.dev`).

```jsx
import React, { useState } from 'react';

const PredictionForm = () => {
  // Estado para os campos do formulário
  const [formData, setFormData] = useState({
    c_date: '2023-01-01', // Valor inicial
    campaign_name: 'facebook_tier1',
    category: 'social',
    impressions: 100000,
    mark_spent: 3000.00,
    clicks: 1000,
    leads: 50,
    orders: 20,
  });

  // Estado para a URL da API (substitua pela sua URL ngrok)
  const [apiUrl, setApiUrl] = useState('https://scintillant-samuel-dowily.ngrok-free.dev/predict');
  // ^^^ COLOQUE SUA URL PÚBLICA DO NGROK AQUI ^^^^

  // Estado para a previsão, loading e erros
  const [prediction, setPrediction] = useState(null);
  const [loading, setLoading] = useState(false);
  const [error, setError] = useState(null);

  const handleChange = (e) => {
    const { name, value } = e.target;
    setFormData(prevData => ({
      ...prevData,
      [name]: value
    }));
  };

  const handleSubmit = async (e) => {
    e.preventDefault();
    setLoading(true);
    setError(null);
    setPrediction(null);

    try {
      // Converter tipos numéricos para float/int antes de enviar
      const dataToSend = {
        ...formData,
        impressions: parseInt(formData.impressions),
        mark_spent: parseFloat(formData.mark_spent),
        clicks: parseInt(formData.clicks),
        leads: parseInt(formData.leads),
        orders: parseInt(formData.orders),
      };

      const response = await fetch(apiUrl, {
        method: 'POST',
        headers: {
          'Content-Type': 'application/json',
        },
        body: JSON.stringify(dataToSend),
      });

      if (!response.ok) {
        const errorData = await response.json();
        throw new Error(errorData.error || `Erro de rede: ${response.status}`);
      }

      const result = await response.json();
      setPrediction(result.predictions[0]);

    } catch (err) {
      console.error('Erro ao fazer a predição:', err);
      setError(`Falha na predição: ${err.message}`);
    } finally {
      setLoading(false);
    }
  };

  return (
    <div style={{ padding: '20px', maxWidth: '600px', margin: 'auto' }}>
      <h1>Previsão de Receita de Marketing</h1>
      <p>Insira os dados da campanha para obter uma previsão de receita.</p>

      <form onSubmit={handleSubmit} style={{ display: 'grid', gridTemplateColumns: '1fr 1fr', gap: '10px' }}>
        <div>
          <label>Data:</label>
          <input type="date" name="c_date" value={formData.c_date} onChange={handleChange} required />
        </div>
        <div>
          <label>Nome da Campanha:</label>
          <input type="text" name="campaign_name" value={formData.campaign_name} onChange={handleChange} required />
        </div>
        <div>
          <label>Categoria:</label>
          <input type="text" name="category" value={formData.category} onChange={handleChange} required />
        </div>
        <div>
          <label>Impressões:</label>
          <input type="number" name="impressions" value={formData.impressions} onChange={handleChange} required />
        </div>
        <div>
          <label>Gasto de Marketing:</label>
          <input type="number" name="mark_spent" step="0.01" value={formData.mark_spent} onChange={handleChange} required />
        </div>
        <div>
          <label>Cliques:</label>
          <input type="number" name="clicks" value={formData.clicks} onChange={handleChange} required />
        </div>
        <div>
          <label>Leads:</label>
          <input type="number" name="leads" value={formData.leads} onChange={handleChange} required />
        </div>
        <div>
          <label>Pedidos:</label>
          <input type="number" name="orders" value={formData.orders} onChange={handleChange} required />
        </div>
        <div style={{ gridColumn: '1 / -1' }}>
          <button type="submit" disabled={loading} style={{ width: '100%', padding: '10px', backgroundColor: 'blue', color: 'white', border: 'none', borderRadius: '5px', cursor: 'pointer' }}>
            {loading ? 'Prevendo...' : 'Obter Previsão'}
          </button>
        </div>
      </form>

      {error && <p style={{ color: 'red' }}>Erro: {error}</p>}

      {prediction !== null && (
        <div style={{ marginTop: '20px', padding: '15px', border: '1px solid green', borderRadius: '5px', backgroundColor: '#e6ffe6' }}>
          <h2>Previsão de Receita:</h2>
          <p style={{ fontSize: '1.5em', fontWeight: 'bold', color: 'green' }}>${prediction.toFixed(2)}</p>
        </div>
      )}
    </div>
  );
};

export default PredictionForm;
```

### Como usar este código:
1.  **Crie um Projeto React**: Se você ainda não tem um, crie um novo projeto React usando `npx create-react-app my-flask-frontend` ou similar.
2.  **Copie o Código**: Salve o código acima como `PredictionForm.jsx` dentro da pasta `src/components/` do seu projeto React.
3.  **Atualize `App.js`**: Modifique seu `src/App.js` para importar e renderizar o `PredictionForm`:
    ```jsx
    import React from 'react';
    import PredictionForm from './components/PredictionForm'; // Ajuste o caminho conforme necessário

    function App() {
      return (
        <div className="App">
          <PredictionForm />
        </div>
      );
    }

    export default App;
    ```
4.  **Inicie o Servidor de Desenvolvimento React**: No terminal do seu projeto React, execute `npm start`.
5.  **Substitua a URL do ngrok**: **Muito importante:** Na linha `const [apiUrl, setApiUrl] = useState('YOUR_NGROK_PUBLIC_URL_HERE/predict');` dentro de `PredictionForm.jsx`, substitua `YOUR_NGROK_PUBLIC_URL_HERE` pela URL que o ngrok gerou para você na saída da célula `a1641fa0` (ex: `https://scintillant-samuel-dowily.ngrok-free.dev`).

Agora você terá um formulário React funcional que envia dados para a sua API Flask em execução no Colab e exibe as previsões.

```markdown
Para integrar com a API Flask a partir de um componente React, você precisará criar ou modificar um arquivo `.jsx` (por exemplo, `PredictionForm.jsx`) em seu projeto React. Este código não pode ser executado diretamente aqui no Colab, mas é o modelo que você deve usar em seu ambiente de desenvolvimento React.

---

## `src/components/PredictionForm.jsx` (Exemplo de Implementação)

Crie ou modifique o arquivo `src/components/PredictionForm.jsx` com o seguinte código. Certifique-se de substituir `YOUR_NGROK_PUBLIC_URL_HERE` pela URL pública do ngrok que foi gerada na célula anterior (por exemplo, `https://scintillant-samuel-dowily.ngrok-free.dev`).

```jsx
import React, { useState } from 'react';

const PredictionForm = () => {
  // Estado para os campos do formulário
  const [formData, setFormData] = useState({
    c_date: '2023-01-01', // Valor inicial
    campaign_name: 'facebook_tier1',
    category: 'social',
    impressions: 100000,
    mark_spent: 3000.00,
    clicks: 1000,
    leads: 50,
    orders: 20,
  });

  // Estado para a URL da API (substitua pela sua URL ngrok)
  const [apiUrl, setApiUrl] = useState('https://scintillant-samuel-dowily.ngrok-free.dev/predict');
  // ^^^ COLOQUE SUA URL PÚBLICA DO NGROK AQUI ^^^^

  // Estado para a previsão, loading e erros
  const [prediction, setPrediction] = useState(null);
  const [loading, setLoading] = useState(false);
  const [error, setError] = useState(null);

  const handleChange = (e) => {
    const { name, value } = e.target;
    setFormData(prevData => ({
      ...prevData,
      [name]: value
    }));
  };

  const handleSubmit = async (e) => {
    e.preventDefault();
    setLoading(true);
    setError(null);
    setPrediction(null);

    try {
      // Converter tipos numéricos para float/int antes de enviar
      const dataToSend = {
        ...formData,
        impressions: parseInt(formData.impressions),
        mark_spent: parseFloat(formData.mark_spent),
        clicks: parseInt(formData.clicks),
        leads: parseInt(formData.leads),
        orders: parseInt(formData.orders),
      };

      const response = await fetch(apiUrl, {
        method: 'POST',
        headers: {
          'Content-Type': 'application/json',
        },
        body: JSON.stringify(dataToSend),
      });

      if (!response.ok) {
        const errorData = await response.json();
        throw new Error(errorData.error || `Erro de rede: ${response.status}`);
      }

      const result = await response.json();
      setPrediction(result.predictions[0]);

    } catch (err) {
      console.error('Erro ao fazer a predição:', err);
      setError(`Falha na predição: ${err.message}`);
    } finally {
      setLoading(false);
    }
  };

  return (
    <div style={{ padding: '20px', maxWidth: '600px', margin: 'auto' }}>
      <h1>Previsão de Receita de Marketing</h1>
      <p>Insira os dados da campanha para obter uma previsão de receita.</p>

      <form onSubmit={handleSubmit} style={{ display: 'grid', gridTemplateColumns: '1fr 1fr', gap: '10px' }}>
        <div>
          <label>Data:</label>
          <input type="date" name="c_date" value={formData.c_date} onChange={handleChange} required />
        </div>
        <div>
          <label>Nome da Campanha:</label>
          <input type="text" name="campaign_name" value={formData.campaign_name} onChange={handleChange} required />
        </div>
        <div>
          <label>Categoria:</label>
          <input type="text" name="category" value={formData.category} onChange={handleChange} required />
        </div>
        <div>
          <label>Impressões:</label>
          <input type="number" name="impressions" value={formData.impressions} onChange={handleChange} required />
        </div>
        <div>
          <label>Gasto de Marketing:</label>
          <input type="number" name="mark_spent" step="0.01" value={formData.mark_spent} onChange={handleChange} required />
        </div>
        <div>
          <label>Cliques:</label>
          <input type="number" name="clicks" value={formData.clicks} onChange={handleChange} required />
        </div>
        <div>
          <label>Leads:</label>
          <input type="number" name="leads" value={formData.leads} onChange={handleChange} required />
        </div>
        <div>
          <label>Pedidos:</label>
          <input type="number" name="orders" value={formData.orders} onChange={handleChange} required />
        </div>
        <div style={{ gridColumn: '1 / -1' }}>
          <button type="submit" disabled={loading} style={{ width: '100%', padding: '10px', backgroundColor: 'blue', color: 'white', border: 'none', borderRadius: '5px', cursor: 'pointer' }}>
            {loading ? 'Prevendo...' : 'Obter Previsão'}
          </button>
        </div>
      </form>

      {error && <p style={{ color: 'red' }}>Erro: {error}</p>}

      {prediction !== null && (
        <div style={{ marginTop: '20px', padding: '15px', border: '1px solid green', borderRadius: '5px', backgroundColor: '#e6ffe6' }}>
          <h2>Previsão de Receita:</h2>
          <p style={{ fontSize: '1.5em', fontWeight: 'bold', color: 'green' }}>${prediction.toFixed(2)}</p>
        </div>
      )}
    </div>
  );
};

export default PredictionForm;
```

### Como usar este código:
1.  **Crie um Projeto React**: Se você ainda não tem um, crie um novo projeto React usando `npx create-react-app my-flask-frontend` ou similar.
2.  **Copie o Código**: Salve o código acima como `PredictionForm.jsx` dentro da pasta `src/components/` do seu projeto React.
3.  **Atualize `App.js`**: Modifique seu `src/App.js` para importar e renderizar o `PredictionForm`:
    ```jsx
    import React from 'react';
    import PredictionForm from './components/PredictionForm'; // Ajuste o caminho conforme necessário

    function App() {
      return (
        <div className="App">
          <PredictionForm />
        </div>
      );
    }

    export default App;
    ```
4.  **Inicie o Servidor de Desenvolvimento React**: No terminal do seu projeto React, execute `npm start`.
5.  **Substitua a URL do ngrok**: **Muito importante:** Na linha `const [apiUrl, setApiUrl] = useState('YOUR_NGROK_PUBLIC_URL_HERE/predict');` dentro de `PredictionForm.jsx`, substitua `YOUR_NGROK_PUBLIC_URL_HERE` pela URL que o ngrok gerou para você na saída da célula `a1641fa0` (ex: `https://scintillant-samuel-dowily.ngrok-free.dev`).

Agora você terá um formulário React funcional que envia dados para a sua API Flask em execução no Colab e exibe as previsões.
```


## Final Task

### Subtask:
Sumarizar os passos para a criação do aplicativo React, destacando a integração com o modelo de ML via API e a exibição das previsões.


## Summary:

### Data Analysis Key Findings
*   **React Environment Setup**: The process began by providing detailed, step-by-step instructions for initializing a new React project using Vite (e.g., `npm create vite@latest meu-app-react -- --template react`), navigating into the project, installing dependencies, and starting a development server.
*   **Form Component Creation**: A React component named `PredictionForm.jsx` was developed to serve as the user interface for inputting prediction variables. This component includes:
    *   Input fields for all eight required variables: `c_date`, `campaign_name`, `category`, `impressions`, `mark_spent`, `clicks`, `leads`, and `orders`.
    *   Use of React's `useState` hooks for managing form state and ensuring reactivity.
    *   Basic inline styling for a presentable user interface.
    *   An `handleSubmit` function as a placeholder for API integration, initially logging form data to the console.
*   **API Integration in React**: The `PredictionForm.jsx` component was further enhanced to integrate with the Flask API:
    *   The `handleSubmit` function was updated to use the `fetch` API for sending `POST` requests to a specified `apiUrl` (e.g., `https://scintillant-samuel-dowily.ngrok-free.dev/predict`).
    *   Form data is converted to JSON and sent with the appropriate `Content-Type: application/json` header.
    *   Numerical input values (`impressions`, `mark_spent`, `clicks`, `leads`, `orders`) are explicitly parsed as integers or floats before being sent to the API to ensure correct data types.
    *   State variables (`loading`, `error`, `prediction`) were introduced to provide user feedback during the API call, displaying messages like "Prevendo..." or any error encountered, and finally the predicted revenue (e.g., `\$${prediction.toFixed(2)}`).
    *   Detailed instructions were provided for the user to replace a placeholder URL with their actual ngrok public URL and integrate the `PredictionForm` component into their `App.jsx` file.

### Insights or Next Steps
*   The development process provides a robust front-end structure for interacting with a machine learning model via API, showcasing essential React concepts like state management, controlled components, and asynchronous data fetching.
*   The next logical step would be to deploy both the Flask API and the React application to a production environment, ensuring the API URL is updated accordingly and that the application is optimized for performance and security.
